# 04 · Orchestrate — 02 Choosing which source to try first

**`04-retrieve/08-backend-cascade.ipynb` tries `pinecone -> pgvector -> local`. It tries them in that order for a biology question, a clinical question, and a question about yesterday's news, because the order is written into the function body. This notebook makes that order a decision the question gets a say in.**

`01-should-i-retrieve.ipynb` decided *whether* to retrieve. This decides
*where from, first*. It is the same move one level down: the cascade stays
exactly as it is — try sources in order, fall through when one comes back
empty — but the order itself is computed per question rather than hard-coded.

The fixed-order cascade is kept and run side by side throughout, because the
contrast is the lesson. A fixed order is not wrong; it is just a choice made
once, at authoring time, for every question that will ever arrive.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `SOURCES` | Three offline stand-in backends, each with a domain tag and a cost/latency estimate | `SOURCES["clinical_index"]["cost_usd"]` |
| `classify_domains` | Rule-based domain tags for a question, same style as notebook 01's cue lists | `classify_domains("what does the trial say?")` → `{"clinical"}` |
| `choose_order` | The decision: domain match first, then cheapest, then fastest | `choose_order(clinical_q)` |
| `probe` | One source lookup, recording cost and latency actually spent | `probe("web_search", q)` |
| `cascade` | Tries an order until a source clears the score floor; returns results plus a probe log | `cascade(order, q)` |
| `FIXED_ORDER` | The always-the-same order, kept for contrast | `cascade(FIXED_ORDER, q)` |

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 1 — three sources that differ in what they hold and what they cost

Each source is an in-memory document list, scored with the same
deterministic hash embedding the rest of this repo uses — no account, no
network. What matters here is not the scoring but the two fields a fixed
cascade has nowhere to put: which domains a source is good for, and what
one probe of it costs in money and milliseconds.

The cost and latency numbers are **estimates written into this notebook**,
not measurements of any real backend. They are here to show a decision
being made from them; swapping in measured numbers changes the orderings
below and nothing else.

In [ ]:
import hashlib
import math
import time


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


SOURCES = {
    "course_store": {
        "domains": {"course"},
        "cost_usd": 0.0,        # local, in-memory: free
        "latency_ms": 5,
        "docs": [
            "Mitochondria perform oxidative phosphorylation, converting nutrients into ATP.",
            "Photosynthesis converts light energy into chemical energy stored in glucose.",
        ],
    },
    "clinical_index": {
        "domains": {"clinical"},
        "cost_usd": 0.0002,     # hosted vector index: a fraction of a cent per query
        "latency_ms": 120,
        "docs": [
            "Early excision and grafting reduces mortality in major pediatric burns.",
            "Split-thickness skin grafts are preferred for large surface areas.",
        ],
    },
    "web_search": {
        "domains": {"news", "recency"},
        "cost_usd": 0.0010,     # metered search API: the most expensive leg
        "latency_ms": 800,
        "docs": [
            "Conference keynote schedules were published this week for the autumn season.",
            "A municipal transit strike affected commuter rail service on Tuesday.",
        ],
    },
}

for _s in SOURCES.values():
    _s["embeddings"] = [hash_embed(d) for d in _s["docs"]]

nbio.table(
    [(name, ",".join(sorted(s["domains"])), f"${s['cost_usd']:.4f}", f"{s['latency_ms']} ms", len(s["docs"]))
     for name, s in SOURCES.items()],
    ("source", "domains", "cost/probe", "latency/probe", "docs"),
)

## Step 2 — reading a domain off the question

The same kind of rule as notebook 01's cue lists, and the same honesty
about it: these patterns recognise what they were given. A question using
none of this vocabulary gets no tags, which is a real case — Step 4 shows
what the ordering does when nothing matches, and it is not an error.

In [ ]:
import re

DOMAIN_CUES = {
    "course": [r"\bmitochondri", r"\bphotosynthes", r"\batp\b", r"\bcell\b",
               r"\bin (this|the) course\b", r"\blecture\b", r"\bsyllabus\b"],
    "clinical": [r"\bburn", r"\bgraft", r"\bpediatric\b", r"\bmortality\b",
                 r"\bpatient", r"\btrial\b", r"\bclinical\b"],
    "news": [r"\bnews\b", r"\bstrike\b", r"\bschedule\b", r"\bannounce"],
    "recency": [r"\b(today|yesterday|this week|latest|recent|as of)\b", r"\b20\d\d\b"],
}
DOMAIN_CUES = {k: [re.compile(p) for p in v] for k, v in DOMAIN_CUES.items()}


def classify_domains(question: str) -> set[str]:
    q = (question or "").lower()
    return {domain for domain, pats in DOMAIN_CUES.items() if any(p.search(q) for p in pats)}


course_q = "How do mitochondria produce ATP?"
clinical_q = "Does early excision reduce mortality in pediatric burns?"
unknown_q = "How should I structure my weekend?"

for q in (course_q, clinical_q, unknown_q):
    print(f"  {sorted(classify_domains(q)) or '(no domain matched)'} <- {q!r}")

assert classify_domains(course_q) == {"course"}
assert classify_domains(clinical_q) == {"clinical"}
assert classify_domains(unknown_q) == set()

## Step 3 — `choose_order`: the decision itself

Three keys, in priority order, applied to every source:

1. **Domain match.** A source tagged for this question's domain goes first.
2. **Cost.** Among equally-relevant sources, the cheaper probe goes first.
3. **Latency.** Among equally relevant and equally priced, the faster one.

That is the whole policy, and it is deliberately small enough to read in
one sitting. The point is not that this ranking function is the right one —
it is that there *is* a ranking function, evaluated per question, instead
of an order frozen in the body of `retrieve()`.

In [ ]:
def choose_order(question: str) -> list[str]:
    """Source names, best-first for this particular question."""
    tags = classify_domains(question)

    def sort_key(name: str) -> tuple:
        s = SOURCES[name]
        domain_match = 0 if (s["domains"] & tags) else 1   # 0 sorts first
        return (domain_match, s["cost_usd"], s["latency_ms"], name)

    return sorted(SOURCES, key=sort_key)


FIXED_ORDER = ["web_search", "clinical_index", "course_store"]

print("fixed order, every question :", FIXED_ORDER)
print()
print("course question  ->", choose_order(course_q))
print("clinical question->", choose_order(clinical_q))
print("unmatched        ->", choose_order(unknown_q))

order_course, order_clinical = choose_order(course_q), choose_order(clinical_q)
assert order_course != order_clinical, "two different questions must produce two different orderings"
assert order_course[0] == "course_store"
assert order_clinical[0] == "clinical_index"
assert choose_order(unknown_q) == sorted(SOURCES, key=lambda n: (SOURCES[n]["cost_usd"], SOURCES[n]["latency_ms"], n)), (
    "with no domain match the order falls back to cheapest-first, which is a decision too"
)
print()
print("confirmed: the order is a function of the question, and degrades to cheapest-first when nothing matches")

## Step 4 — `probe` and `cascade`: the same fall-through, now over a chosen order

`cascade` is structurally the same loop as
`04-retrieve/08-backend-cascade.ipynb`'s `retrieve`: try a source, and if
what came back is not good enough, fall through to the next one. Two
things are added. The order is a parameter rather than three hard-coded
calls, and every probe is logged with the cost and latency it spent — so
"the cascade worked" and "the cascade worked after paying for two useless
probes first" stop looking the same.

`SCORE_FLOOR` is what counts as good enough to stop. A hash embedding has
no notion of synonymy, so a genuinely on-topic query against its own
source clears it comfortably while an off-topic one does not — the floor
is printed below rather than asserted as a universal constant, because it
is tuned to this offline stand-in, not to any real embedding model.

In [ ]:
SCORE_FLOOR = 0.12


def probe(source_name: str, query: str, top_k: int = 2) -> dict:
    """One source lookup, with what it cost to make."""
    s = SOURCES[source_name]
    time.sleep(s["latency_ms"] / 1000 / 100)  # scaled down 100x so the notebook stays quick
    qvec = hash_embed(query)
    hits = [{"source": source_name, "text": d, "score": cosine(qvec, e)}
            for d, e in zip(s["docs"], s["embeddings"])]
    hits.sort(key=lambda h: -h["score"])
    return {"source": source_name, "hits": hits[:top_k],
            "best_score": hits[0]["score"] if hits else 0.0,
            "cost_usd": s["cost_usd"], "latency_ms": s["latency_ms"]}


def cascade(order: list[str], query: str) -> dict:
    """Try sources in the given order, stopping at the first that clears the floor."""
    log, results = [], []
    for name in order:
        r = probe(name, query)
        log.append(r)
        if r["best_score"] >= SCORE_FLOOR:
            results = r["hits"]
            break
    return {
        "results": results,
        "answered_by": log[-1]["source"] if results else None,
        "probes": len(log),
        "cost_usd": sum(r["cost_usd"] for r in log),
        "latency_ms": sum(r["latency_ms"] for r in log),
        "log": log,
    }


demo = cascade(choose_order(course_q), course_q)
print(f"score floor: {SCORE_FLOOR}")
nbio.table([(r["source"], f"{r['best_score']:.3f}", "cleared" if r["best_score"] >= SCORE_FLOOR else "fell through")
            for r in demo["log"]],
           ("probed", "best score", "verdict"))
print(f"\nanswered by: {demo['answered_by']} after {demo['probes']} probe(s)")

## Step 5 — the same two questions, chosen order against fixed order

Both cascades find the same answer. That is the honest framing: the fixed
order is not broken, it is *wasteful*, and only on the questions whose
right source happens to sit at the end of it. The table shows what the
waste actually is — extra probes, extra cents, extra milliseconds, on
every question of that shape, forever.

In [ ]:
rows = []
for label, q in [("course question", course_q), ("clinical question", clinical_q)]:
    chosen = cascade(choose_order(q), q)
    fixed = cascade(FIXED_ORDER, q)
    rows.append((label, "chosen", " -> ".join(choose_order(q))[:44], chosen["probes"],
                 f"${chosen['cost_usd']:.4f}", f"{chosen['latency_ms']}ms", str(chosen["answered_by"])))
    rows.append((label, "fixed", " -> ".join(FIXED_ORDER)[:44], fixed["probes"],
                 f"${fixed['cost_usd']:.4f}", f"{fixed['latency_ms']}ms", str(fixed["answered_by"])))

nbio.table(rows, ("question", "policy", "order tried", "probes", "cost", "latency", "answered by"))

for q in (course_q, clinical_q):
    chosen, fixed = cascade(choose_order(q), q), cascade(FIXED_ORDER, q)
    assert chosen["answered_by"] == fixed["answered_by"], (
        "both policies must reach the same source — the choice changes the route, not the answer"
    )
    assert chosen["probes"] <= fixed["probes"]
    assert chosen["cost_usd"] <= fixed["cost_usd"]
    assert chosen["latency_ms"] <= fixed["latency_ms"]

course_chosen, course_fixed = cascade(choose_order(course_q), course_q), cascade(FIXED_ORDER, course_q)
assert course_chosen["probes"] < course_fixed["probes"], (
    "for a course question the fixed order pays for web_search and clinical_index first"
)
print()
print(f"confirmed: same answer, {course_fixed['probes'] - course_chosen['probes']} fewer probes "
      f"and ${course_fixed['cost_usd'] - course_chosen['cost_usd']:.4f} less spent on the course question")

## Step 6 — where the choice is worth nothing

A decision layer that only ever looks good is a decision layer that has
not been tested honestly. Two cases where choosing the order buys nothing:

- A question whose best source already sits first in the fixed order. The
  chosen order and the fixed order agree, probe counts are identical, and
  the rule work was pure overhead.
- A question no source can answer. Every order probes everything and finds
  nothing; ordering changes only which useless probe was paid for first.

Both are asserted below, so neither can quietly stop being true.

In [ ]:
news_q = "Was there a transit strike this week?"          # web_search is first in FIXED_ORDER already
nothing_q = "What is the melting point of gallium nitride?"  # in none of the three sources

news_chosen, news_fixed = cascade(choose_order(news_q), news_q), cascade(FIXED_ORDER, news_q)
none_chosen, none_fixed = cascade(choose_order(nothing_q), nothing_q), cascade(FIXED_ORDER, nothing_q)

nbio.table(
    [("news question", news_chosen["probes"], news_fixed["probes"], str(news_chosen["answered_by"])),
     ("answerable by nothing", none_chosen["probes"], none_fixed["probes"], str(none_chosen["answered_by"]))],
    ("case", "probes (chosen)", "probes (fixed)", "answered by"),
)

assert news_chosen["probes"] == news_fixed["probes"] == 1, (
    "when the fixed order already leads with the right source, choosing buys nothing"
)
assert none_chosen["results"] == [] and none_fixed["results"] == []
assert none_chosen["probes"] == none_fixed["probes"] == len(SOURCES), (
    "an unanswerable question costs a full sweep under either policy"
)
print()
print("confirmed: the decision layer is worth nothing in both of these cases, and costs a rule evaluation to find out")

## What did not come across

- **The cost and latency numbers are written, not measured.** They are
  plausible orders of magnitude for a local store, a hosted index and a
  metered search API; they are not benchmarks of any deployed backend.
  Real numbers belong in a config file that this policy reads, and
  producing them is measurement work, not orchestration work.
- **The domain rules are a lexicon, not a classifier.** Same limitation
  as `01-should-i-retrieve.ipynb`: they recognise the vocabulary they were
  given. `unknown_q` in Step 3 matches nothing at all, and the fallback —
  cheapest first — is a reasonable default rather than a right answer.
- **No source is ever tried twice, and no results are merged across
  sources.** This cascade stops at the first source clearing the floor,
  exactly as `04-retrieve/08-backend-cascade.ipynb` does. Merging across
  several sources is `04-retrieve/02-multi-source-fanout.ipynb`'s job;
  merging across several *attempts* is the next notebook's.
- **Nothing here checks whether what came back was any good.** The score
  floor is a retrieval score, not a judgment about the answer. That check
  — and acting on it — is `03-retry-on-verdict.ipynb`.